# Breast Cancer Detection - Exploratory Data Analysis

**Student ID:** 10953361  
**Course:** PUSL3190 Computing Project  
**Dataset:** Wisconsin Breast Cancer Dataset (sklearn)

---

## Objective
This notebook performs comprehensive exploratory data analysis on the breast cancer dataset to:
- Understand data distribution and characteristics
- Identify important features for classification
- Compare multiple machine learning models
- Provide insights for the Flask web application design

## Section 1 — Dataset Loading & Overview

In [ ]:
# Import required libraries
from sklearn.datasets import load_breast_cancer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully")

In [ ]:
# Load the breast cancer dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print(f"Dataset Shape: {df.shape}")
print(f"Number of samples: {df.shape[0]}")
print(f"Number of features: {df.shape[1] - 1}")
print(f"\nTarget classes: {data.target_names}")

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Dataset information
df.info()

### Key Observations:
- **569 samples** with **30 features** + 1 target column
- All features are numerical (float64)
- Features represent measurements of cell nuclei from digitized images
- Target: 0 = Malignant, 1 = Benign

## Section 2 — Class Distribution

In [ ]:
# Count of each class
class_counts = df['target'].value_counts()
print("Class Distribution:")
print(f"Benign (1): {class_counts[1]} samples ({class_counts[1]/len(df)*100:.1f}%)")
print(f"Malignant (0): {class_counts[0]} samples ({class_counts[0]/len(df)*100:.1f}%)")

In [ ]:
# Visualize class distribution
plt.figure(figsize=(8, 6))
colors = ['#ef4444', '#16a34a']
df['target'].value_counts().plot(kind='bar', color=colors, edgecolor='black', linewidth=1.5)
plt.title('Benign vs Malignant Distribution', fontsize=16, fontweight='bold')
plt.xticks([0, 1], ['Malignant (0)', 'Benign (1)'], rotation=0, fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xlabel('Class', fontsize=12)
plt.grid(axis='y', alpha=0.3)

# Add count labels on bars
for i, v in enumerate(class_counts.values):
    plt.text(i, v + 5, str(v), ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

### Key Observations:
- **357 Benign** cases (62.7%)
- **212 Malignant** cases (37.3%)
- Dataset is **slightly imbalanced** but not severely
- Stratified sampling will be used during train-test split

## Section 3 — Missing Value Check

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing Values per Column:")
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "✅ No missing values found!")
print(f"\nTotal missing values: {missing_values.sum()}")

### Key Observations:
- **Zero missing values** across all features
- No imputation or data cleaning required
- Dataset is ready for modeling without preprocessing for missing data

## Section 4 — Feature Correlation Heatmap

In [ ]:
# Compute correlation matrix
correlation_matrix = df.corr()

# Plot heatmap
plt.figure(figsize=(20, 16))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0,
            linewidths=0.5, cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Feature Correlation Heatmap', fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Find highly correlated features with target
target_corr = correlation_matrix['target'].sort_values(ascending=False)
print("Top 10 Features Correlated with Target:")
print(target_corr[1:11])  # Exclude target itself

### Key Observations:
- Many features show **high correlation** with each other (multicollinearity)
- **Worst features** (worst radius, worst perimeter, worst area) show strongest correlation with target
- **Mean concave points** and **worst concave points** are highly predictive
- Feature selection or dimensionality reduction could improve model performance

## Section 5 — Feature Distribution by Class

In [ ]:
# Select key features for visualization
features = ['mean radius', 'mean texture', 'mean perimeter',
            'mean area', 'mean smoothness', 'mean concavity']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, feature in enumerate(features):
    ax = axes[idx]
    
    # Plot KDE for each class
    df[df['target'] == 0][feature].plot(kind='kde', ax=ax, label='Malignant', color='#ef4444', linewidth=2)
    df[df['target'] == 1][feature].plot(kind='kde', ax=ax, label='Benign', color='#16a34a', linewidth=2)
    
    ax.set_title(feature.title(), fontsize=14, fontweight='bold')
    ax.set_xlabel('Value', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.legend(loc='best')
    ax.grid(alpha=0.3)

plt.suptitle('Feature Distribution by Class (KDE)', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for worst features
worst_features = ['worst radius', 'worst texture', 'worst perimeter',
                  'worst area', 'worst concavity', 'worst concave points']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, feature in enumerate(worst_features):
    ax = axes[idx]
    df.boxplot(column=feature, by='target', ax=ax, patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='black'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(feature.title(), fontsize=14, fontweight='bold')
    ax.set_xlabel('Class (0=Malignant, 1=Benign)', fontsize=11)
    ax.set_ylabel('Value', fontsize=11)
    plt.sca(ax)
    plt.xticks([1, 2], ['Malignant', 'Benign'])

plt.suptitle('Worst Features Distribution by Class (Box Plots)', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

### Key Observations:
- **Clear separation** between benign and malignant classes for most features
- **Malignant tumors** tend to have higher values for radius, perimeter, area, and concavity
- **Worst features** show better class separation than mean features
- Features like **worst concave points** and **worst radius** are highly discriminative

## Section 6 — Model Training & Comparison

In [ ]:
# Import ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("✅ ML libraries imported")

In [ ]:
# Prepare data
X = df.drop('target', axis=1)
y = df['target']

# Split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nClass distribution in training set:")
print(y_train.value_counts())
print(f"\nClass distribution in test set:")
print(y_test.value_counts())

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Features scaled using StandardScaler")
print(f"Mean of scaled training data: {X_train_scaled.mean():.6f}")
print(f"Std of scaled training data: {X_train_scaled.std():.6f}")

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42)
}

# Train and evaluate
results = {}

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Training {name}...")
    print(f"{'='*60}")
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    
    # Evaluate
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    
    print(f"\n{name} Accuracy: {accuracy*100:.2f}%")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print(f"\nConfusion Matrix:")
    print(cm)

In [ ]:
# Visualize model comparison
plt.figure(figsize=(10, 6))
model_names = list(results.keys())
accuracies = [results[name] * 100 for name in model_names]

bars = plt.bar(model_names, accuracies, color=['#1A73E8', '#16A34A', '#DC2626'],
               edgecolor='black', linewidth=1.5)
plt.title('Model Accuracy Comparison', fontsize=16, fontweight='bold')
plt.ylabel('Accuracy (%)', fontsize=12)
plt.xlabel('Model', fontsize=12)
plt.ylim(90, 100)
plt.grid(axis='y', alpha=0.3)

# Add accuracy labels on bars
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{acc:.1f}%', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

# Print best model
best_model = max(results, key=results.get)
print(f"\n🏆 Best Model: {best_model} with {results[best_model]*100:.2f}% accuracy")

### Key Observations:
- All three models achieve **>95% accuracy**
- **SVM** typically performs best (~97-98%)
- **Random Forest** and **Logistic Regression** also perform well
- High precision and recall for both classes
- Models are production-ready for the Flask application

## Section 7 — SHAP Visualization (Feature Importance)

In [ ]:
# Install SHAP if not available
try:
    import shap
    print("✅ SHAP library loaded")
except ImportError:
    print("Installing SHAP...")
    !pip install shap
    import shap
    print("✅ SHAP installed and loaded")

In [ ]:
# Create SHAP explainer for Logistic Regression
lr_model = models['Logistic Regression']
explainer = shap.LinearExplainer(lr_model, X_train_scaled)
shap_values = explainer.shap_values(X_test_scaled)

print("✅ SHAP values computed")

In [ ]:
# Summary plot (bar) - Global feature importance
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.title('Global Feature Importance (SHAP)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Summary plot (beeswarm) - Feature impact distribution
plt.figure(figsize=(10, 10))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('Feature Impact Distribution (SHAP)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Get top 10 most important features
feature_importance = np.abs(shap_values).mean(axis=0)
feature_names = X.columns
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

print("\nTop 10 Most Important Features (SHAP):")
print(importance_df.head(10))

### Key Observations:
- **Worst concave points** is the most important feature globally
- **Worst perimeter** and **worst radius** are also highly influential
- **Mean concave points** and **mean concavity** contribute significantly
- SHAP values show both magnitude and direction of feature impact
- These insights guide the Flask system's feature prioritization

## Section 8 — Conclusion & Summary

In [ ]:
print("="*70)
print("EXPLORATORY DATA ANALYSIS - SUMMARY OF FINDINGS")
print("="*70)

print("\n📊 DATASET CHARACTERISTICS:")
print(f"   • Total samples: {df.shape[0]}")
print(f"   • Total features: {df.shape[1] - 1}")
print(f"   • Missing values: 0 (100% complete data)")
print(f"   • Data types: All numerical (float64)")

print("\n🎯 CLASS DISTRIBUTION:")
print(f"   • Benign cases: {class_counts[1]} ({class_counts[1]/len(df)*100:.1f}%)")
print(f"   • Malignant cases: {class_counts[0]} ({class_counts[0]/len(df)*100:.1f}%)")
print(f"   • Balance: Slightly imbalanced but acceptable")

print("\n🤖 MODEL PERFORMANCE:")
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"   • {name}: {acc*100:.2f}%")
print(f"   • Best model: {best_model}")

print("\n⭐ MOST IMPORTANT FEATURES (Top 5):")
for idx, row in importance_df.head(5).iterrows():
    print(f"   {idx+1}. {row['Feature']}: {row['Importance']:.4f}")

print("\n💡 KEY INSIGHTS FOR FLASK SYSTEM:")
print("   • All three models (LR, RF, SVM) are production-ready")
print("   • Feature scaling (StandardScaler) is essential")
print("   • Worst features (radius, perimeter, concave points) are most predictive")
print("   • SHAP explanations provide interpretability for medical professionals")
print("   • No data preprocessing needed (no missing values)")
print("   • Stratified sampling ensures balanced train/test splits")

print("\n✅ CONCLUSION:")
print("   The Wisconsin Breast Cancer dataset is well-suited for machine learning.")
print("   High accuracy (>95%) achieved with multiple models. The Flask web")
print("   application will use these trained models to provide real-time predictions")
print("   with SHAP-based explanations for clinical decision support.")

print("\n" + "="*70)
print("END OF ANALYSIS")
print("="*70)

---

## References

1. **Dataset Source:**  
   Dua, D. and Graff, C. (2019). UCI Machine Learning Repository. University of California, Irvine, School of Information and Computer Sciences.  
   [https://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Wisconsin+(Diagnostic)](https://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Wisconsin+(Diagnostic))

2. **Original Paper:**  
   W.N. Street, W.H. Wolberg and O.L. Mangasarian. Nuclear feature extraction for breast tumor diagnosis. IS&T/SPIE 1993 International Symposium on Electronic Imaging: Science and Technology, volume 1905, pages 861-870, San Jose, CA, 1993.

3. **SHAP Library:**  
   Lundberg, S. M., & Lee, S. I. (2017). A unified approach to interpreting model predictions. Advances in neural information processing systems, 30.

---

**Student ID:** 10953361  
**Course:** PUSL3190 Computing Project  
**Date:** 2026

---